# 09. Niepewność TTA per wycinek: wszystkie seedy i rundy (MoNuSeg, D4)

**Pytanie:** czy zmiany predykcji studenta po transformacjach D4 wskazują miejsca, w których student się
myli, liczone na **wycinkach** obrazu (250 px, siatka 4×4), a nie na całych obrazach.

**Dane:** wyniki `uncertainty/tile_scores.py` dla studentów random AL: 6 seedów × rundy 2, 4, 6, 8, 10, 12
(liczba oznaczonych obrazów), na obrazach nieoznaczonej puli i walidacji (test nietknięty). Dla każdego
wycinka: miary studenta (`K_mean`, `unstable`, `U_MI_mean`, `Top_mean`) i trudność $u = 1 - PQ$ studenta
względem wyroczni (tylko do oceny). Notebook 08 robił to samo dla 1 studenta i 9 obrazów.

### Skąd są studenci i jak to policzono

- **Studentów nie trenowaliśmy na potrzeby tej analizy.** To gotowe checkpointy z wcześniejszego przebiegu
  random AL na MoNuSeg (`outputs/al_random/full/seed<k>/round_XX/models/student`, rcmc). Każdy student to
  Cellpose-SAM z enkoderem z oryginalnego SAM ViT-L, trenowany od zera w swojej rundzie na maskach wyroczni
  oznaczonych obrazów: 200 epok × 64 losowe fragmenty 256 px, batch 4, lr 5·10⁻⁵, weight decay 0,1
  (3200 kroków, tyle samo w każdej rundzie). Seed zmienia tylko kolejność losowania obrazów.
- **Tu jest tylko inferencja.** Dla każdego studenta (6 seedów × rundy 2–12) `uncertainty/tile_scores.py`:
  1. segmentuje każdy obraz puli (nieoznaczony w tej rundzie) i walidacji w 8 wersjach D4 (obroty o 90°,
     odbicia), mapuje predykcje z powrotem na oryginał,
  2. liczy miary jąder (`tta.nucleus_uncertainty`: K, D, Top, U_MI, ...) względem predykcji oryginału,
  3. dzieli obraz na 16 wycinków 250 px (jądro należy do wycinka swojego środka) i agreguje miary
     (`patches.tile_uncertainty`),
  4. liczy PQ studenta względem wyroczni per wycinek (`patches.tile_quality`; dopasowanie na całym obrazie),
  5. zapisuje `tiles.csv`, `images.csv`, `nuclei.csv` do `outputs/tile_scores/al_random_full/seed<k>/round_XX/`.
- **Gdzie:** A100 na rcmc-gpu; skrypt sterujący na rcmc kopiował checkpointy jednego seeda, liczył, sprawdzał
  pliki i usuwał kopie. Ok. 5,3 s na obraz, 17 min na seed, 1 h 42 min łącznie.
- Ten notebook tylko czyta te tabele (CPU).

**Oceny:**
- **A.** ρ Spearmana wynik wycinka – $u$ **wewnątrz obrazu** (średnia po obrazach), osobno dla rund; 95% CI
  bootstrapem **po obrazach** (wszystkie seedy i rundy danego obrazu losowane razem). Do porównania:
  AUC na poziomie jąder (jądro błędne = brak odpowiednika u wyroczni z IoU > 0,5).
- **B.** Czy wynik obrazu złożony z wycinków (najgorszy, 2 najgorsze) szereguje obrazy puli lepiej niż
  średnia po jądrach (ρ z $u$ obrazu, liczone w każdym seedzie i rundzie).
- **C.** Domknięta luka do podglądu przy wyborze `k` wycinków z puli vs wyborze całych obrazów
  (32 wycinki = 2 obrazy, ten sam budżet pikseli). To nie jest pętla AL, tylko trafność rankingu.

In [ ]:
MIN_NUCLEI = 5  # wycinki z mniejszą liczbą jąder studenta pomijamy (szum)
N_BOOT = 1000  # powtórzenia bootstrap po obrazach
SCORES = ["rank_kd", "unstable", "K_mean", "Top_mean", "U_MI_mean", "n_nuclei"]
K_TILES = (8, 16, 32, 64)  # 32 wycinki = 2 obrazy, 64 = 4 obrazy

In [ ]:
%matplotlib inline
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from histopathology_pnw.config import Settings

warnings.filterwarnings("ignore", message="An input array is constant")

SRC = Settings().outputs_dir / "tile_scores" / "al_random_full"
OUT = Settings().outputs_dir / "tile_eval_full"
OUT.mkdir(parents=True, exist_ok=True)


def load(name):
    files = sorted(SRC.glob(f"seed*/round_??/{name}"))
    return pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


tiles, images, nuclei = load("tiles.csv"), load("images.csv"), load("nuclei.csv")
use = tiles[(tiles["n_nuclei"] >= MIN_NUCLEI) & tiles["u"].notna()].copy()
KEY = ["seed", "n_labeled", "part"]
# rank_kd jak selektor tta_kd: rangi procentowe K_mean i unstable w obrębie (seed, runda, część)
use["rank_kd"] = use.groupby(KEY)[["K_mean", "unstable"]].rank(pct=True).mean(axis=1)
images["rank_kd"] = images.groupby(KEY)[["K_mean", "unstable"]].rank(pct=True).mean(axis=1)
ROUNDS = sorted(tiles["n_labeled"].unique())
print(
    f"studenci: {tiles.groupby(['seed', 'n_labeled']).ngroups}, obrazy: {len(images)}, "
    f"wycinki: {len(tiles)} (w analizie {len(use)}), jądra: {len(nuclei)}"
)
images.groupby("n_labeled")[["u", "n_nuclei"]].mean().round(3).T

## A. Czy wynik wycinka wskazuje trudny wycinek wewnątrz obrazu?

Dla każdego obrazu (w danym seedzie i rundzie) liczymy ρ Spearmana między wynikiem wycinka a jego $u$,
potem uśredniamy. Ten sam obraz występuje u wielu studentów (różne seedy i rundy), więc bootstrap losuje
**obrazy**, a nie wycinki ani pary (obraz, student). `n_nuclei` to kontrola: czy trudność nie wynika
po prostu z liczby jąder.

In [ ]:
def within_rho(df, cols, min_tiles=4):
    rows = []
    for key, g in df.groupby([*KEY, "image_id"]):
        if len(g) < min_tiles:
            continue
        r = g[cols].corrwith(g["u"], method="spearman")
        rows.append({**dict(zip([*KEY, "image_id"], key, strict=True)), **r.to_dict()})
    return pd.DataFrame(rows)


R = within_rho(use, SCORES)
rng = np.random.default_rng(0)


def boot_mean(r, col):
    # mean over (image, student) groups; CI by resampling images
    per = r.groupby("image_id")[col].agg(["sum", "count"])
    s, c = per["sum"].to_numpy(), per["count"].to_numpy()
    idx = rng.integers(0, len(per), (N_BOOT, len(per)))
    boots = s[idx].sum(axis=1) / c[idx].sum(axis=1)
    return s.sum() / c.sum(), *np.percentile(boots, [2.5, 97.5])


tab = []
for part in ("pool", "val"):
    for rnd in [*ROUNDS, "wszystkie"]:
        r = (
            R[R["part"] == part]
            if rnd == "wszystkie"
            else R[(R["part"] == part) & (R["n_labeled"] == rnd)]
        )
        for col in SCORES:
            m, lo, hi = boot_mean(r.dropna(subset=[col]), col)
            tab.append({"część": part, "runda": rnd, "wynik": col, "rho": m, "lo": lo, "hi": hi})
tab_a = pd.DataFrame(tab)
tab_a.to_csv(OUT / "A_within_rho.csv", index=False)
show = tab_a[tab_a["runda"] == "wszystkie"].assign(
    CI=lambda d: d.apply(lambda x: f"[{x.lo:.2f}, {x.hi:.2f}]", axis=1)
)
show.pivot(index="wynik", columns="część", values="rho").loc[SCORES].round(3).join(
    show.pivot(index="wynik", columns="część", values="CI").add_suffix(" CI")
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for ax, part in zip(axes, ("pool", "val"), strict=True):
    for col in SCORES:
        d = tab_a[(tab_a["część"] == part) & (tab_a["runda"] != "wszystkie")]
        d = d[d["wynik"] == col]
        x = d["runda"].astype(int)
        ax.plot(x, d["rho"], marker="o", label=col)
        ax.fill_between(x, d["lo"], d["hi"], alpha=0.12)
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_xlabel("runda (liczba oznaczonych obrazów)")
    ax.set_title(f"A: rho wewnątrz obrazu, {part}")
axes[0].set_ylabel("Spearman wynik wycinka - u")
axes[1].legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT / "A_within_rho_by_round.png", dpi=110, bbox_inches="tight")

### Odniesienie: poziom jąder

AUC (jądro błędne vs poprawne) dla każdej rundy, liczone na wszystkich jądrach puli i walidacji.
Jądro błędne = najlepsze IoU z jądrem wyroczni ≤ 0,5. Dla `D` kierunek jest odwrotny (1 − D).

In [ ]:
def auc(score, err):
    s = pd.Series(score).rank().to_numpy()
    n1, n0 = err.sum(), (~err).sum()
    return (s[err].sum() - n1 * (n1 + 1) / 2) / (n1 * n0) if n1 and n0 else np.nan


nuclei["err"] = nuclei["iou_oracle"] <= 0.5
MEAS = {
    "K": nuclei["K"],
    "1 - D": 1 - nuclei["D"],
    "U_MI": nuclei["U_MI"],
    "Top": nuclei["Top"],
    "U_H": nuclei["U_H"],
    "pole (mniejsze = gorzej)": -nuclei["area"],
}
rows = []
for rnd, g in nuclei.groupby("n_labeled"):
    e = g["err"].to_numpy()
    rows.append(
        {
            "runda": rnd,
            "jądra": len(g),
            "błędne": e.mean(),
            **{k: auc(v.loc[g.index], e) for k, v in MEAS.items()},
        }
    )
tab_auc = pd.DataFrame(rows)
tab_auc.to_csv(OUT / "nucleus_auc.csv", index=False)
tab_auc.round(3)

## B. Wynik obrazu złożony z wycinków

W każdym seedzie i rundzie, na obrazach **puli**: ρ Spearmana między wynikiem obrazu a $u$ obrazu.
- `średnia jąder`: średnia po wszystkich jądrach obrazu (jak selektor `tta_kd`),
- `max wycinka`, `top2 wycinków`: najgorszy wycinek, średnia z 2 najgorszych (wycinki z ≥ `MIN_NUCLEI` jądrami).

Średnia ± odchylenie standardowe po 6 seedach.

In [ ]:
def from_tiles(col, how):
    def f(g):
        v = g[col].dropna().sort_values(ascending=False)
        return v.iloc[0] if how == "max" else v.iloc[:2].mean()

    return use.groupby([*KEY, "image_id"]).apply(f).rename(f"{col}|{how}")


pool_img = images[images["part"] == "pool"].set_index([*KEY, "image_id"])
for col in ("K_mean", "unstable"):
    for how in ("max", "top2"):
        pool_img = pool_img.join(from_tiles(col, how))
rows = []
for (seed, rnd), g in pool_img.groupby(level=["seed", "n_labeled"]):
    for col in ("K_mean", "unstable", "rank_kd"):
        variants = {"średnia jąder": col}
        if col != "rank_kd":
            variants |= {"max wycinka": f"{col}|max", "top2 wycinków": f"{col}|top2"}
        for name, c in variants.items():
            rows.append(
                {
                    "seed": seed,
                    "runda": rnd,
                    "miara": col,
                    "wynik obrazu": name,
                    "rho": g[c].corr(g["u"], method="spearman"),
                }
            )
tab_b = pd.DataFrame(rows)
tab_b.to_csv(OUT / "B_image_from_tiles.csv", index=False)
summary_b = tab_b.groupby(["miara", "wynik obrazu", "runda"])["rho"].mean().unstack("runda")
summary_b["wszystkie"] = tab_b.groupby(["miara", "wynik obrazu"])["rho"].mean()
summary_b["sd seedów"] = (
    tab_b.groupby(["miara", "wynik obrazu", "seed"])["rho"].mean().groupby(level=[0, 1]).std()
)
summary_b.round(3)

## C. Wybór wycinków vs wybór obrazów (offline)

W każdym seedzie i rundzie, w **puli**: selektor bierze `k` wycinków (albo obrazów) z najwyższym wynikiem,
podgląd bierze `k` o najwyższym $u$, losowy ma w oczekiwaniu średnie $u$ puli.

$$\text{domknięta luka} = \frac{\bar u_{\text{selektor}} - \bar u_{\text{pula}}}{\bar u_{\text{podgląd}} - \bar u_{\text{pula}}}$$

Wycinki i obrazy porównujemy przy tym samym budżecie pikseli (32 wycinki = 2 obrazy, 64 = 4 obrazy).
`max 4 / obraz`: wybór wycinków z limitem 4 na obraz (więcej różnych obrazów w wyborze).

In [ ]:
def gap(df, col, k, cap=None):
    pool = df["u"].mean()
    order = df.sort_values(col, ascending=False)
    if cap:
        order = order.groupby("image_id").head(cap)
    sel = order.head(k)["u"].mean()
    best = df.nlargest(k, "u")["u"].mean()
    return (sel - pool) / (best - pool) if best > pool else np.nan


rows = []
for (seed, rnd), g in use[use["part"] == "pool"].groupby(["seed", "n_labeled"]):
    gi = images[
        (images["part"] == "pool") & (images["seed"] == seed) & (images["n_labeled"] == rnd)
    ]
    for col in ("rank_kd", "unstable", "K_mean", "U_MI_mean"):
        r = {"seed": seed, "runda": rnd, "wynik": col}
        for k in K_TILES:
            r[f"wycinki k={k}"] = gap(g, col, k)
        r["wycinki k=32, max 4 / obraz"] = gap(g, col, 32, cap=4)
        r["obrazy k=2"] = gap(gi, col, 2)
        r["obrazy k=4"] = gap(gi, col, 4)
        rows.append(r)
tab_c = pd.DataFrame(rows)
tab_c.to_csv(OUT / "C_selection_gap.csv", index=False)
cols_c = [c for c in tab_c.columns if c.startswith(("wycinki", "obrazy"))]
tab_c.groupby("wynik")[cols_c].mean().loc[["rank_kd", "unstable", "K_mean", "U_MI_mean"]].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.2))
d = tab_c[tab_c["wynik"] == "rank_kd"].groupby("runda")
for c, style in (
    ("wycinki k=32", "-o"),
    ("wycinki k=32, max 4 / obraz", "-s"),
    ("obrazy k=2", "--o"),
):
    m, s = d[c].mean(), d[c].std()
    ax.errorbar(m.index, m, yerr=s, fmt=style, capsize=3, label=c)
ax.axhline(0, color="gray", lw=0.8)
ax.set_xlabel("runda (liczba oznaczonych obrazów)")
ax.set_ylabel("domknięta luka (0 = losowy, 1 = podgląd)")
ax.set_title("C: rank_kd, ten sam budżet pikseli (średnia ± sd po seedach)")
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(OUT / "C_gap_by_round.png", dpi=110, bbox_inches="tight")

### C2. To samo w jednej skali: średnie $u$ wybranych pikseli

Znormalizowana luka nie porównuje uczciwie wycinków z obrazami: każda jednostka ma swój własny podgląd
(32 najtrudniejsze wycinki są dużo trudniejsze niż 2 najtrudniejsze obrazy). Tu liczymy wprost średnie
$u$ **wycinków** w wybranym budżecie: dla wyboru obrazów bierzemy wszystkie wycinki wybranych 2 obrazów,
dla wyboru wycinków 32 wybrane wycinki (ten sam budżet pikseli). Wyższe = selekcja trafia w więcej błędów.

In [ ]:
rows = []
for (seed, rnd), g in use[use["part"] == "pool"].groupby(["seed", "n_labeled"]):
    gi = images[
        (images["part"] == "pool") & (images["seed"] == seed) & (images["n_labeled"] == rnd)
    ]

    def tiles_of(ids, g=g):
        return g[g["image_id"].isin(ids)]["u"].mean()

    rows.append(
        {
            "seed": seed,
            "runda": rnd,
            "pula (losowy)": g["u"].mean(),
            "obrazy k=2 (rank_kd)": tiles_of(gi.nlargest(2, "rank_kd")["image_id"]),
            "wycinki k=32 (rank_kd)": g.nlargest(32, "rank_kd")["u"].mean(),
            "podgląd: obrazy k=2": tiles_of(gi.nlargest(2, "u")["image_id"]),
            "podgląd: wycinki k=32": g.nlargest(32, "u")["u"].mean(),
        }
    )
tab_c2 = pd.DataFrame(rows)
tab_c2.to_csv(OUT / "C2_selected_u.csv", index=False)
cols_c2 = [c for c in tab_c2.columns if c not in ("seed", "runda")]
better = (tab_c2["wycinki k=32 (rank_kd)"] > tab_c2["obrazy k=2 (rank_kd)"]).mean()
print(f"wycinki lepsze od obrazów w {better:.0%} z {len(tab_c2)} par (seed, runda)")
out = tab_c2.groupby("runda")[cols_c2].mean()
out.loc["wszystkie"] = tab_c2[cols_c2].mean()
out.round(3)

## Wnioski

**A. Niestabilność po transformacjach wskazuje trudne miejsca wewnątrz obrazu.** Na 36 studentach
(6 seedów × rundy 2–12) ρ wewnątrz obrazu wynosi ok. 0,57 dla `rank_kd` (CI ok. [0,53; 0,61] w puli),
ok. 0,51 dla `K_mean` i ok. 0,50 dla `unstable`; tak samo w walidacji i we wszystkich rundach.
Liczba jąder w wycinku nie tłumaczy trudności (ρ ≈ 0). `U_MI` (0,33) i `Top` (0,19) są słabsze.
Na poziomie jąder AUC dla K wynosi 0,86–0,88 i lekko rośnie z rundami; `U_MI` jest tu równie dobre,
ale po uśrednieniu na wycinek traci (tak jak na poziomie obrazu u Mateusza).

**B. Wynik obrazu z najgorszych wycinków jest gorszy od średniej po jądrach** (np. `K_mean`: 0,69 dla
średniej, 0,51 dla 2 najgorszych wycinków, 0,44 dla najgorszego). Do szeregowania całych obrazów
lepsza jest średnia; wynik notebooka 08 się potwierdza.

**C. Przy tym samym budżecie pikseli wybór wycinków trafia w trochę więcej błędów niż wybór obrazów**
(C2: średnie $u$ 0,413 vs 0,401, w ok. 80% par seed–runda) i dorównuje podglądowi wyroczni dla całych
obrazów. Sufit dla wycinków jest dużo wyższy (podgląd 0,47 vs 0,415), więc selekcja per wycinek ma
więcej miejsca na poprawę. Limit 4 wycinków na obraz obniża trafność.